In [2]:
import os
import sys
import numpy as np
import time
from collections import defaultdict,deque
from typing import List, Optional, Tuple, Union
from scipy.interpolate import CubicSpline

sys.path.insert(0, os.getcwd())  # run this notebook from the hybrid-mesh-adaptation/ folder so the meshAdapt/ package here is importable
from meshAdapt import Triangulation, aflrAdapt, exportMesh, blMesh, unitVec, gammaMGamma, searchTriangle, baryCentricCoordinates

In [3]:
def read_mesh(mesh_file: str, metric_file: Optional[str] = None) -> Union[
        Tuple[List[List[int]], List[List[int]], List[List[float]]],
        Tuple[List[List[int]], List[List[int]], List[List[float]], List[List[float]]]
    ]:
    """
    Read a 2D triangular mesh file.

    Parameters
    ----------
    mesh_file : str
        Path to mesh file.
    metric_file : Optional[str]
        Path to metric file (optional). If provided, metric values
        are read and returned.

    Returns
    -------
    element_vertex_id : List[List[int]]
        element_vertex_id[k] = [n0, n1, n2]
        Triangle k made of vertices n0, n1, n2 (0-based indexing)

    boundary_data : {}
        boundary_data[i] = []
        Boundary edge with surface ID and vertex indices (0-based)

    vertex_coords : List[List[float]]
        vertex_coords[i] = [x, y]
        Coordinates of vertex i

    metric_values : List[List[float]], optional
        metric_values[i] = [m11, m12, m22] (or appropriate components)
        Metric tensor components at vertex i.
        Returned only if metric_file is provided.
    """

    with open(mesh_file, 'r') as f:
        lines = f.readlines()

    def find_section(keyword: str) -> int:
        """
        Return the line index of `keyword` in the mesh file.

        Parameters
        ----------
        keyword : str
            Section header string (e.g. 'surfaceelements').

        Returns
        -------
        int
            Zero-based index of the line that reads `keyword\n`.

        Raises
        ------
        ValueError
            If the keyword is not found in the file.
        """
        try:
            return lines.index(keyword + '\n')
        except ValueError:
            raise ValueError(f"Keyword '{keyword}' not found in mesh file")

    # --- surface elements ---
    idx = find_section('surfaceelements')
    nelm = int(lines[idx + 1].strip())

    element_vertex_id = []
    for i in range(nelm):
        tokens = lines[idx + 2 + i].split()
        np_elm = int(tokens[4])  # tokens: surfnr bcnr domin domout np_elm n0 n1 ...
        if len(tokens) < 5 + np_elm:
            raise ValueError(f"Invalid surface element line at {idx + 2 + i}")
        if np_elm == 3:
            element_vertex_id.append([int(tokens[5]) - 1, int(tokens[6]) - 1, int(tokens[7]) - 1])
        elif np_elm == 4:
            element_vertex_id.append([int(tokens[5]) - 1, int(tokens[6]) - 1, int(tokens[7]) - 1, int(tokens[8]) - 1])
        else:
            raise ValueError(f"Unsupported element with np={np_elm}")

    # --- boundary edges ---
    idx = find_section('edgesegmentsgi2')
    nbdry = int(lines[idx + 1].strip())

    boundary_data = {}
    for i in range(nbdry):
        tokens = lines[idx + 2 + i].split()
        parsed = []
        for val in tokens:
            try:
                parsed.append(int(val))
            except ValueError:
                parsed.append(float(val))
        boundary_data[i] = parsed
        boundary_data[i][2] -= 1  # convert 1-based node indices to 0-based
        boundary_data[i][3] -= 1

    # --- vertex coordinates ---
    idx = find_section('points')
    nvert = int(lines[idx + 1].strip())

    vertex_coords = []
    for i in range(nvert):
        tokens = lines[idx + 2 + i].split()
        vertex_coords.append([float(tokens[0]), float(tokens[1])])

    # --- optional metric file ---
    if metric_file is not None:
        metric_values = []
        with open(metric_file, 'r') as f:
            next(f)
            for line in f:
                tokens = line.split()
                metric_values.append([float(val) for val in tokens])
        return element_vertex_id, boundary_data, vertex_coords, metric_values

    return element_vertex_id, boundary_data, vertex_coords

def tri_area(v1, v2, v3):
    """
    Compute the area of a triangle given its three vertex coordinates.

    Uses the 2-D cross-product (shoelace) formula.

    Parameters
    ----------
    v1, v2, v3 : tuple of float
        (x, y) coordinates of each vertex.

    Returns
    -------
    float
        Triangle area (always non-negative).
    """
    x1, y1 = v1
    x2, y2 = v2
    x3, y3 = v3
    area = 0.5 * abs((x2 - x1) * (y3 - y1) - (x3 - x1) * (y2 - y1))
    return area

def quads_to_tris(quads):
    """
    Split each quad element into two triangles along the 0-2 diagonal.

    Parameters
    ----------
    quads : list[list[int]]
        Quad elements, each [n0, n1, n2, n3] with 0-based node IDs.

    Returns
    -------
    list[list[int]]
        Triangle elements; each quad produces [n0, n1, n2] and [n0, n2, n3].
    """
    triangles = []
    for quad in quads:
        n0, n1, n2, n3 = quad
        triangles.append([n0, n1, n2])
        triangles.append([n0, n2, n3])
    return triangles

def compute_normals(viscous_edges_id, vertex_coords):
    """
    Compute averaged unit outward normals at every node of the viscous boundary.

    For each edge the outward normal is the 90-degree CCW rotation of the edge
    tangent, normalised to unit length.  At nodes shared by multiple edges the
    normals are accumulated and re-normalised to give a smooth corner normal.

    Parameters
    ----------
    viscous_edges_id : list[list[int]]
        Ordered list of [n0, n1] boundary edges.
    vertex_coords : list[list[float]]
        Vertex coordinates [x, y].

    Returns
    -------
    dict
        avgNormals[node_id] = [nx, ny] â€” unit outward normal at that node.
    """
    avgNormals = {}
    for edge in viscous_edges_id:
        n0, n1 = edge
        x0, y0 = vertex_coords[n0]
        x1, y1 = vertex_coords[n1]
        ex = x1 - x0
        ey = y1 - y0
        nx = -ey  # 90Â° CCW rotation of edge tangent (ex,ey) -> (-ey,ex) gives outward normal
        ny = ex
        length = (nx**2 + ny**2)**0.5
        if length > 0:
            nx /= length
            ny /= length
        for n in (n0, n1):
            if n not in avgNormals:
                avgNormals[n] = [nx, ny]
            else:
                avgNormals[n][0] += nx
                avgNormals[n][1] += ny
    # re-normalize accumulated normals at nodes shared by multiple edges
    for n in avgNormals:
        nx, ny = avgNormals[n]
        length = (nx**2 + ny**2)**0.5
        if length > 0:
            avgNormals[n][0] /= length
            avgNormals[n][1] /= length
    return avgNormals

def order_boundary_data(boundary_data, bcFlag):
    """
    Re-order boundary_data so that edges for `bcFlag` form a contiguous chain.

    The matching edges are sorted into a connected sequence and placed at the
    front of the returned dict; all other boundary entries follow unchanged.

    Parameters
    ----------
    boundary_data : dict
        As returned by read_mesh; values are parsed boundary-edge records.
    bcFlag : int
        BC surface ID whose edges should be chained and placed first.

    Returns
    -------
    dict
        New boundary_data with bcFlag edges ordered into a chain at the front,
        followed by all other boundary entries in their original order.
    """
    # separate bcFlag edges from all other boundary records
    edges = []
    edge_to_full = {}
    other_entries = []

    for val in boundary_data.values():
        if val[0] == bcFlag:
            n0, n1 = val[2], val[3]
            edges.append([n0, n1])
            edge_to_full[(n0, n1)] = val
            edge_to_full[(n1, n0)] = val
        else:
            other_entries.append(val)

    # adjacency map: start_node -> edges departing from it
    start_map = defaultdict(list)
    for e in edges:
        start_map[e[0]].append(e)

    # find the chain start: a node that no edge points toward
    end_nodes = set(e[1] for e in edges)
    start = None
    for e in edges:
        if e[0] not in end_nodes:
            start = e[0]
            break
    if start is None:
        start = edges[0][0]

    # follow the chain head-to-tail
    ordered_edges = []
    current = start
    while current in start_map and start_map[current]:
        e = start_map[current].pop(0)
        ordered_edges.append(e)
        current = e[1]

    # bcFlag edges first (in traversal order), then all other entries
    new_boundary_data = {}
    idx = 0
    for e in ordered_edges:
        new_boundary_data[idx] = edge_to_full[(e[0], e[1])]
        idx += 1
    for val in other_entries:
        new_boundary_data[idx] = val
        idx += 1

    return new_boundary_data

def quad_layer(bgMesh, viscous_edges_id, farfield_edges_id, vertex_coords, blLayers, blHeight,
               delta0=None, growthFactor=None, closed=True, normal_sign=1.0, nvert_background=None,
               protected_nodes=None):
    """
    Generate structured quad boundary layers using advancing front + deformation.

    Parameters
    ----------
    bgMesh : object
        Background mesh providing metric via bgMesh.metric(x, y)

    viscous_edges_id : list[list[int]]
        Ordered list of boundary edges. A closed loop (e.g. an airfoil
        surface) when `closed=True`; an open chain with distinct start/end
        nodes (e.g. a shock curve) when `closed=False`.

    vertex_coords : list[list[float]]
        List of vertex coordinates [x, y]; will be modified in-place

    blLayers : int
        Maximum number of boundary layers

    blHeight : float
        Maximum total height of boundary layer

    closed : bool
        Whether `viscous_edges_id` forms a closed loop. When False, the
        wraparound quad/edge that stitches the last node back to the first
        is skipped (there is no such connection on an open chain).

    normal_sign : float
        +1.0 (default) advances along the normals as computed by
        compute_normals; -1.0 advances along the opposite (flipped) normal.
        Use this to grow layers on the other side of an open curve, e.g.
        the "inside" of a shock curve after building the "outside" with
        +1.0 -- see shock_quad_layers below.

    nvert_background : int, optional
        Overrides the count of nodes treated as "background mesh" for the
        IDW deformation step (default: len(vertex_coords) at call time).
        Pass the same fixed value to multiple chained calls so a later call
        doesn't treat an earlier call's newly created layer nodes as
        background and deform them.

    protected_nodes : iterable[int], optional
        Extra node ids to exclude from the IDW deformation step, beyond
        base_nodes_set/farfield_nodes_set. Needed when chaining calls (e.g.
        shock_quad_layers): nvert_background alone only stops a later call
        from touching the EARLIER call's node range, but nodes from the
        earlier call's own base_nodes (which now hold its finished outer
        layer, not generic background) fall inside that same low-numbered
        range and must be named here explicitly.

    Returns
    -------
    vertex_coords : list
        Updated vertex list including newly created nodes

    new_quads : list[list[int]]
        List of generated quad elements

    updated_viscous_edges : list[list[int]]
        New boundary edges after first layer extrusion

    blLayerHeights : list[float]
        Heights of each generated layer
"""

    # ------------------------------------------------------------------
    # Step 0: Extract ordered boundary node list
    # ------------------------------------------------------------------
    base_nodes = [viscous_edges_id[0][0]]
    for edge in viscous_edges_id:
        base_nodes.append(edge[1])
    if closed:
        base_nodes.pop()  # remove duplicate closing node

    base_nodes_set = set(base_nodes)  # faster membership check

    farfield_nodes_set = set(n for edge in farfield_edges_id for n in edge)
    extra_protected_set = set(protected_nodes) if protected_nodes else set()

    # nvert_background lets callers freeze which nodes count as the
    # original background mesh across multiple chained quad_layer calls
    # (e.g. shock_quad_layers growing both sides of the same curve) --
    # without it, a second chained call would treat the first call's
    # newly created layer nodes as background and deform them.
    nvert_old = nvert_background if nvert_background is not None else len(vertex_coords)
    totalHeight = 0.0
    nlayers = 0

    new_layer = {}         # new_layer[k] = node ids at layer k
    new_quads = []
    blLayerHeights = []

    # ------------------------------------------------------------------
    # Main advancing loop
    # ------------------------------------------------------------------
    while nlayers < blLayers and totalHeight < blHeight:

        new_layer[nlayers] = []
        nvert = len(vertex_coords)

        normals = compute_normals(viscous_edges_id, vertex_coords)
        original_coords = [coord[:] for coord in vertex_coords]

        layer_height = []

        if delta0 is not None and growthFactor is not None:
            height = delta0 * growthFactor**nlayers
        else:
            for node in base_nodes:
                x, y = vertex_coords[node]
                metric = bgMesh.metric(x, y)
                dy = 1.0 / gammaMGamma(normals[node], metric)
                layer_height.append(dy)
            min_layer_heigth = min(layer_height)
            avg_layer_height = sum(layer_height) / len(layer_height)
            t = 0.0
            height = (1-t) * min_layer_heigth + t * avg_layer_height

        if totalHeight + height > blHeight:
            break

        blLayerHeights.append(height)

        for nn, node in enumerate(base_nodes):
            x, y = vertex_coords[node]
            nx, ny = normals[node]
            vertex_coords[node][0] = x + normal_sign * height * nx
            vertex_coords[node][1] = y + normal_sign * height * ny
            new_node_id = nvert + nn
            new_layer[nlayers].append(new_node_id)
            vertex_coords.append([x, y])

        totalHeight += height
        nlayers += 1

        L_def = 1.0
        alpha = 1.0
        a = 10
        b = 10
        for v in range(nvert_old):
            if v in base_nodes_set or v in farfield_nodes_set or v in extra_protected_set:
                continue
            xv0 = np.array(original_coords[v])
            num = np.zeros(2)
            den = 0.0
            for i in base_nodes:
                xsi0 = np.array(original_coords[i])
                si = np.array(vertex_coords[i]) - xsi0
                r = np.linalg.norm(xv0 - xsi0)
                w = (L_def / r)**a + (alpha * L_def / r)**b
                num += w * si
                den += w
            if den > 1e-14:
                vertex_coords[v] = list(xv0 + num / den)

    new_layer[nlayers] = base_nodes

    n_base = len(base_nodes)
    for k in range(nlayers):
        layer_k = new_layer[k]
        layer_k1 = new_layer[k + 1]
        for i in range(n_base - 1):
            new_quads.append([layer_k[i], layer_k[i + 1], layer_k1[i + 1], layer_k1[i]])
        if closed:
            new_quads.append([layer_k[-1], layer_k[0], layer_k1[0], layer_k1[-1]])

    updated_viscous_edges = [[new_layer[0][i], new_layer[0][i + 1]] for i in range(n_base - 1)]
    if closed:
        updated_viscous_edges.append([new_layer[0][-1], new_layer[0][0]])

    return vertex_coords, new_quads, updated_viscous_edges, blLayerHeights


def shock_quad_layers(bgMesh, shock_edges_id, farfield_edges_id, vertex_coords,
                       n_layers_outside, n_layers_inside,
                       blHeight_outside=float("inf"), blHeight_inside=float("inf"),
                       delta0=None, growthFactor=None):
    """
    Build structured quad layers on BOTH sides of an open shock curve.

    Calls quad_layer twice with closed=False: once advancing along the
    normals computed by compute_normals ("outside", normal_sign=+1.0) and
    once along the flipped normals ("inside", normal_sign=-1.0).

    The inside pass is chained off the outside pass's `updated_viscous_edges`
    (the fresh copy of the original curve nodes that quad_layer creates for
    exactly this kind of chaining) rather than the original shock_edges_id,
    so it grows from nodes still sitting at the curve's original position --
    the outside pass's own base nodes end up holding its *outermost* layer
    and must not be touched again. Both calls share one frozen
    `nvert_background` (the vertex count before either call) so neither
    pass's deformation step treats the other pass's new layer nodes as
    background mesh to be smoothed.

    Parameters
    ----------
    shock_edges_id : list[list[int]]
        Ordered OPEN chain of edges along the shock curve (distinct start
        and end nodes, not a closed loop).

    n_layers_outside, n_layers_inside : int
        Number of layers to grow on each side (caller-specified).

    blHeight_outside, blHeight_inside : float
        Optional cap on total layer height per side; default unbounded so
        n_layers_outside / n_layers_inside is the controlling stop
        condition.

    Returns
    -------
    vertex_coords : list
        Updated vertex list including all newly created nodes from both sides

    quads_outside, quads_inside : list[list[int]]
        Generated quad elements on each side

    edges_outside, edges_inside : list[list[int]]
        New boundary edges after the first layer on each side (i.e. still
        running along the original curve position, one layer in)

    heights_outside, heights_inside : list[float]
        Per-layer heights on each side
    """
    nvert_background = len(vertex_coords)

    vertex_coords, quads_outside, edges_outside, heights_outside = quad_layer(
        bgMesh, shock_edges_id, farfield_edges_id, vertex_coords,
        n_layers_outside, blHeight_outside, delta0, growthFactor,
        closed=False, normal_sign=1.0, nvert_background=nvert_background,
    )

    # protect every node belonging to the finished outside structure (its
    # original base nodes -- now holding the outermost layer -- plus every
    # intermediate layer node) from the inside pass's deformation step
    protect_outside = set(n for quad in quads_outside for n in quad)

    vertex_coords, quads_inside, edges_inside, heights_inside = quad_layer(
        bgMesh, edges_outside, farfield_edges_id, vertex_coords,
        n_layers_inside, blHeight_inside, delta0, growthFactor,
        closed=False, normal_sign=-1.0, nvert_background=nvert_background,
        protected_nodes=protect_outside,
    )

    return (vertex_coords, quads_outside, quads_inside,
            edges_outside, edges_inside, heights_outside, heights_inside)

In [5]:
def read_in2d_wall_geometry(in2d_file, bc_flag=1):
    """
    Parse an .in2d file and return the ordered wall geometry points for bc_flag.
    Returns a list of [x, y] in traversal order.
    Here,only used to read the farfield boundary.
    """
    # open() reads the .in2d file into memory using UTF-8 encoding
    with open(in2d_file, encoding="utf-8") as f:
        # lines: every raw line from the file, used for two-pass parsing
        lines = f.readlines()

    # pts: dict mapping each point ID (int) to its [x, y] coordinates
    pts = {}
    # in_pts: flag that becomes True once the 'points' section header is seen
    in_pts = False
    for line in lines:
        # s: current line stripped of leading/trailing whitespace for clean comparison
        s = line.strip()
        if s == "points":
            in_pts = True
            continue
        if s in ("segments", "materials"):
            in_pts = False
            continue
        if in_pts and s:
            # tokens: whitespace-separated fields of the current point line
            tokens = s.split()
            if len(tokens) >= 3:
                try:
                    pts[int(tokens[0])] = [float(tokens[1]), float(tokens[2])]
                except ValueError:
                    pass

    # edges: list of (n1, n2) node-index pairs for wall segments matching bc_flag
    edges = []
    # in_segs: flag that becomes True once the 'segments' section header is seen
    in_segs = False
    for line in lines:
        s = line.strip()
        if s == "segments":
            in_segs = True
            continue
        if s == "materials":
            break
        if in_segs and "-bc=" in s:
            # tokens: whitespace-separated fields of the segment line
            tokens = s.split()
            # bc_tok: the token(s) carrying the boundary condition tag (e.g. '-bc=2')
            bc_tok = [t for t in tokens if t.startswith("-bc=")]
            if bc_tok and int(bc_tok[0].split("=")[1]) == bc_flag:
                edges.append((int(tokens[3]), int(tokens[4])))

    if not edges:
        return []

    # adj: adjacency dict mapping each start-node to its list of successor nodes
    adj = {}
    # setdefault() initialises a neighbour list for each node if absent, then appends the next node
    for n1, n2 in edges:
        adj.setdefault(n1, []).append(n2)

    # start: the first node ID used as the traversal entry point
    start = edges[0][0]
    # ordered: growing list of node IDs in traversal order
    # visited: set of already-seen node IDs to prevent revisiting
    # cur: the node currently being expanded in the traversal
    ordered, visited, cur = [start], {start}, start
    while True:
        # adj.get() retrieves the neighbours of the current node (empty list if none)
        # nexts: unvisited neighbours of the current node
        nexts = [n for n in adj.get(cur, []) if n not in visited]
        if not nexts:
            break
        # nxt: the next node to step to along the wall
        nxt = nexts[0]
        ordered.append(nxt)
        visited.add(nxt)
        cur = nxt

    # resolve ordered node IDs back to [x, y] coordinates from the pts dict
    return [pts[pid] for pid in ordered]


def parse_in2d_spline_segments(in2d_file, bc_flag=2):
    """
    Parse rational quadratic Bezier (conic) segments from an .in2d splinecurves2dv2 file.

    Each 3-point wall segment has p0 and p2 as curve endpoints and p_ctrl as
    the tangent intersection control point (NOT on the curve).

    Returns list of (p0, p_ctrl, p2) numpy arrays.
    """
    # open() reads the .in2d file into memory using UTF-8 encoding
    # lines: every raw line from the file, used for two-pass parsing
    with open(in2d_file, encoding="utf-8") as f:
        lines = f.readlines()

    # pts: dict mapping each point ID (int) to its [x, y] numpy array
    pts = {}
    # in_pts: flag that becomes True once the "points" section header is seen
    in_pts = False
    for line in lines:
        # s: current line stripped of leading/trailing whitespace for clean comparison
        s = line.strip()
        if s == "points":
            in_pts = True
            continue
        if s in ("segments", "materials"):
            in_pts = False
            continue
        if in_pts and s:
            # tokens: whitespace-separated fields of the current point line
            tokens = s.split()
            if len(tokens) >= 3:
                try:
                    pts[int(tokens[0])] = np.array([float(tokens[1]), float(tokens[2])])
                except ValueError:
                    pass

    # segments: list of (p0, p_ctrl, p2) numpy-array triples for each 3-point wall segment
    segments = []
    # in_segs: flag that becomes True once the "segments" section header is seen
    in_segs = False
    for line in lines:
        s = line.strip()
        if s == "segments":
            in_segs = True
            continue
        if s == "materials":
            break
        if in_segs and "-bc=" in s:
            # tokens: whitespace-separated fields of the segment line
            tokens = s.split()
            # bc_tok: the token(s) carrying the boundary condition tag (e.g. "-bc=2")
            bc_tok = [t for t in tokens if t.startswith("-bc=")]
            if bc_tok and int(bc_tok[0].split("=")[1]) == bc_flag:
                if int(tokens[2]) == 3:
                    segments.append((pts[int(tokens[3])], pts[int(tokens[4])], pts[int(tokens[5])]))

    return segments


def bezier_point(p0, p_ctrl, p2, t):
    """Evaluate rational quadratic Bezier (formula A.3) at parameter t."""
    w_num = np.linalg.norm(p0 - p2)
    w_den = np.sqrt(0.5 * (np.linalg.norm(p0 - p_ctrl)**2 + np.linalg.norm(p2 - p_ctrl)**2))
    w     = w_num / w_den if w_den > 1e-14 else 1.0
    denom = (1 - t)**2 + w * t * (1 - t) + t**2
    return ((1 - t)**2 * p0 + w * t * (1 - t) * p_ctrl + t**2 * p2) / denom


def build_wall_spline_map(spline_segs, quad_boundary, wall_bc):
    """
    Build a map from each wall mesh edge to its Bezier geometry using ednr/dist
    from the .vol boundary records rather than coordinate proximity.

    Each boundary record for a wall edge carries:
      val[8]  ednr1 -- geometry edge number at p1 (1-based)
      val[9]  dist1 -- parametric position of p1 on that edge
      val[10] ednr2 -- geometry edge number at p2
      val[11] dist2 -- parametric position of p2 on that edge

    Returns dict mapping canonical edge key (min_n, max_n) to
    (p0, p_ctrl, p2, dist_a, dist_b) where dist_a is at key[0] and
    dist_b at key[1], so theta = dist_a + t_inner*(dist_b - dist_a)
    gives the exact geometry parameter for any interior split.
    Edges that straddle a geometry segment junction (ednr1 != ednr2)
    are skipped; those are corner points and never need a curved split.
    """
    # spline_segs   : list of (p0, p_ctrl, p2) Bezier triples, one per wall geometry segment
    # quad_boundary : dict of .vol boundary records for all quad mesh edges, keyed by edge
    # wall_bc       : integer BC tag identifying wall edges (e.g. 2)
    spline_map = {}  # output dict: maps (min_n, max_n) edge key to Bezier geometry and parametric extents
    for val in quad_boundary.values():  # val: one .vol boundary record for a quad mesh edge
        if val[0] != wall_bc:
            continue
        n1, n2   = val[2], val[3]  # mesh node IDs at the two endpoints of this wall edge
        ednr1, dist1 = int(val[8]),  float(val[9])  # 1-based geometry segment index and arc-length parameter at n1
        ednr2, dist2 = int(val[10]), float(val[11])  # 1-based geometry segment index and arc-length parameter at n2
        if ednr1 != ednr2:
            continue  # junction edge -- corner node, no curved split needed
        si = ednr1 - 1  # 0-based segment index into spline_segs
        if si < 0 or si >= len(spline_segs):
            continue
        p0, p_ctrl, p2 = spline_segs[si]  # Bezier control points: start, off-curve tangent intersection, end
        key = (min(n1, n2), max(n1, n2))  # canonical sorted edge identifier, direction-independent
        if key[0] == n1:
            spline_map[key] = (p0, p_ctrl, p2, dist1, dist2)
        else:
            spline_map[key] = (p0, p_ctrl, p2, dist2, dist1)
    return spline_map

def split_add(split_list, t, nid, tol=1e-1):
    """Insert (t, nid) into a sorted split list if t is not already present."""
    # split_list: list of (param, node_id) tuples sorted by param, records edge split points
    # t: float parameter value (0.0-1.0) along the edge where the split occurs
    # nid: integer node ID assigned to the new split point
    # tol: float tolerance for treating two t values as the same split
    for et, enid in split_list:  # et: existing param; enid: existing node ID at that param
        if abs(et - t) < tol:  # duplicate split detected within tolerance
            return enid
    split_list.append((t, nid))
    split_list.sort()  # keep list ordered by parameter value
    return nid


def split_get(split_list, t, tol=1e-1):
    """Return nid for the entry at parameter t, or None if not found."""
    for et, enid in split_list:
        if abs(et - t) < tol:
            return enid
    return None


def trace_bl_column(outer_key, t_outer_can, edge_split, edge_to_quads, quad_quads, vertex_coords,
                     wall_spline_map=None):
    """
    Starting from outer_key (an interface edge), trace a split seam inward
    through successive BL quad layers until the wall is reached.

    At each quad the seam enters via the outer edge and exits via the opposite
    (inner) edge.  A new node is inserted on the inner edge at the same
    parametric position as the outer split, and the process repeats for the
    next quad that owns that inner edge.

    For wall edges, if wall_spline_map is provided, the wall node is placed on
    the exact rational quadratic Bezier (formula A.3) using theta = t_inner
    (the seam fraction extended straight from the BL interface to the wall):

        w   = ||p_i - p_j|| / sqrt(0.5*(||p_i - p_ij^t||^2 + ||p_j - p_ij^t||^2))
        p_θ = ((1-θ)^2*p_i + w*θ*(1-θ)*p_ij^t + θ^2*p_j)
              / ((1-θ)^2 + w*θ*(1-θ) + θ^2)

    For all other (non-wall) inner edges, linear interpolation is used.

    Newly created nodes are appended to vertex_coords and recorded in
    edge_split so they can be reused if two traces hit the same inner edge.

    Returns
    -------
    list of (qi, li, outer_nid, inner_nid)
        qi        : quad index
        li        : local edge index of the outer edge inside that quad
        outer_nid : node id of the split on the outer edge
        inner_nid : node id of the new split on the inner edge
    """
    # current_key: the edge key being processed in this iteration of the traversal
    current_key = outer_key
    # current_t: canonical parametric position on current_key (measured from key[0] to key[1])
    current_t   = t_outer_can
    # visited: set of quad indices already processed to prevent revisiting
    visited     = set()
    # records: growing list of (qi, li, outer_nid, inner_nid) tuples describing each split
    records     = []

    while True:
        # candidates: unvisited (quad_idx, local_edge_idx) pairs that own the current edge
        # edge_to_quads is a dict mapping each edge key → list of (quad_index, local_edge_index) pairs, i.e. every quad that owns that edge and which side of the quad that edge is.
        candidates = [
            (qi, li)
            for qi, li in edge_to_quads.get(current_key, [])
            if qi not in visited
        ]
        if not candidates:
            break

        # qi: index of the next quad to process along the seam
        # li: local edge index of current_key within that quad
        qi, li = candidates[0]
        # quad: the node list for this quad element
        quad = quad_quads[qi]
        visited.add(qi)

        # a: the node at the start of the outer edge (at local index li)
        a = quad[li]
        # t_from_a: parametric position measured from node a (flips if a is key[1] not key[0])
        t_from_a = current_t if (a == current_key[0]) else (1.0 - current_t)

        # inner_left: the quad node on the inner edge adjacent to a (opposite corner)
        inner_left  = quad[(li + 3) % 4]
        # inner_right: the quad node on the inner edge adjacent to b (opposite corner)
        inner_right = quad[(li + 2) % 4]

        # p_il: numpy coordinate array for the inner_left node
        p_il = np.array(vertex_coords[inner_left])
        # p_ir: numpy coordinate array for the inner_right node
        p_ir = np.array(vertex_coords[inner_right])

        # inner_key: canonical (min, max) edge key for the inner (wall-side) edge
        inner_key = (min(inner_left, inner_right), max(inner_left, inner_right))
        # t_inner: canonical parametric position on inner_key (from key[0] to key[1])
        t_inner   = t_from_a if (inner_left == inner_key[0]) else (1.0 - t_from_a)

        if inner_key not in edge_split:
            edge_split[inner_key] = []
        if split_get(edge_split[inner_key], t_inner) is None:
            if wall_spline_map is not None and inner_key in wall_spline_map:
                p0, p_ctrl, p2, dist_a, dist_b = wall_spline_map[inner_key]
                theta = dist_a + t_inner * (dist_b - dist_a)
                p_new = bezier_point(p0, p_ctrl, p2, theta).tolist()
            else:
                # p_new: new inner-edge node by linear interpolation at t_from_a
                p_new = (p_il + t_from_a * (p_ir - p_il)).tolist()
            # nid: index of the newly appended node in vertex_coords
            nid = len(vertex_coords)
            vertex_coords.append(p_new)
            split_add(edge_split[inner_key], t_inner, nid)

        # outer_nid: node ID of the split point on the current outer edge
        outer_nid = split_get(edge_split[current_key], current_t)
        # inner_nid: node ID of the split point on the inner edge just computed
        inner_nid = split_get(edge_split[inner_key], t_inner)

        records.append((qi, li, outer_nid, inner_nid, t_from_a))
        current_key = inner_key
        current_t   = t_inner

    return records


def project_and_trace_seams(
    tri_nodes, iface_edges_data, vertex_coords, edge_to_quads, quad_quads,
    corner_tol, wall_spline_map=None,
):
    """Project tri interface nodes onto BL edges, then trace column seams inward."""
    # edge_split: maps each split edge key to a sorted list of (canonical_t, node_id) pairs
    edge_split       = {}
    # tri_node_mapping: maps each tri interface node to its snapped/projected quad node
    tri_node_mapping = {}

    for tn in tri_nodes:
        # p: numpy coordinate array for the current tri interface node
        p = np.array(vertex_coords[tn])
        # best_dist: closest projection distance found so far across all interface edges
        best_dist = np.inf
        # best_t, best_proj, best_a, best_b: projection results for the closest edge so far
        best_t = best_proj = best_a = best_b = None

        for (a, b, pa, pb) in iface_edges_data:
            # ab: edge vector from pa to pb
            ab  = pb - pa
            # ab2: squared length of the edge (used to normalise the projection)
            ab2 = np.dot(ab, ab)
            if ab2 < 1e-12:
                continue
            # t: unclamped parametric projection of p onto the edge line
            t    = np.dot(p - pa, ab) / ab2
            # t_cl: t clamped to [0, 1] so the projection stays on the edge segment
            t_cl = np.clip(t, 0.0, 1.0)
            # proj: the closest point on the edge to the tri node
            proj = pa + t_cl * ab
            # dist: Euclidean distance from the tri node to its projection
            dist = np.linalg.norm(p - proj)
            if dist < best_dist:
                best_dist = dist
                best_t    = t_cl
                best_proj = proj
                best_a, best_b = a, b

        if best_t <= corner_tol:
            tri_node_mapping[tn] = best_a
            vertex_coords[tn]    = vertex_coords[best_a][:]
        elif best_t >= 1.0 - corner_tol:
            tri_node_mapping[tn] = best_b
            vertex_coords[tn]    = vertex_coords[best_b][:]
        else:
            # key: canonical (min, max) edge key for the best-matching interface edge
            key   = (min(best_a, best_b), max(best_a, best_b))
            # t_can: canonical parametric position on key (from key[0] to key[1])
            t_can = best_t if (best_a == key[0]) else (1.0 - best_t)
            if key not in edge_split:
                edge_split[key] = []
            nid = split_get(edge_split[key], t_can)
            if nid is None:
                # nid: index of the new node appended to vertex_coords
                nid = len(vertex_coords)
                vertex_coords.append(best_proj.tolist())
                split_add(edge_split[key], t_can, nid)
            tri_node_mapping[tn] = nid
            vertex_coords[tn]    = vertex_coords[nid][:]
            #e.g.:edge_split[(8,12)]={(0.4,101),(0.7,102)}

            #(8,12)->interface_quad_edge;(0.4,101),(0.7,102)->(t,new_node_id)

    # quad_split: maps each split quad index to (li, sorted [(t_from_a, outer_nid, inner_nid), ...])
    #e.g..:quad_split[2]=(1,[(0.4,])
    quad_split = {}
    for edge_key, splits in list(edge_split.items()):
        if len(edge_to_quads.get(edge_key, [])) != 1:
            continue
        for (t_can, _) in splits:
            # trace_bl_column() follows the split seam inward through successive BL quad layers
            # records: list of (qi, li, outer_nid, inner_nid, t_from_a) from each layer traversed
            records = trace_bl_column(
                edge_key, t_can, edge_split, edge_to_quads, quad_quads, vertex_coords,
                wall_spline_map=wall_spline_map,
            )
            for (qi, li, outer_nid, inner_nid, t_from_a) in records:
                if qi not in quad_split:
                    quad_split[qi] = (li, [])
                seam_list = quad_split[qi][1]
                if not any(abs(t - t_from_a) < 1e-10 for t, _, _ in seam_list):
                    seam_list.append((t_from_a, outer_nid, inner_nid))
                    seam_list.sort()

    return edge_split, tri_node_mapping, quad_split


def build_elements(
    quad_quads, tri_elements, tri_node_mapping, quad_split,
    edge_split, edge_to_quads, interface_edges, vertex_coords,
):
    """Split quads along seams, remap/bisect tris, and return the combined element list."""
    # final_elements: growing list of all output elements (quads first, then tris)
    final_elements = []
    for qi, quad in enumerate(quad_quads):
        if qi not in quad_split:
            final_elements.append(quad)
            continue
        # li: local edge index of the outer (interface) edge within this quad
        # seams: sorted [(t_from_a, outer_nid, inner_nid), ...] for all splits through this quad
        li, seams = quad_split[qi]
        # a: the first node of the outer edge (at local index li)
        a           = quad[li]
        # b: the second node of the outer edge (at local index li+1)
        b           = quad[(li + 1) % 4]
        # inner_right: the inner-edge node adjacent to b
        inner_right = quad[(li + 2) % 4]
        # inner_left: the inner-edge node adjacent to a
        inner_left  = quad[(li + 3) % 4]
        # outer_nodes/inner_nodes: all split nodes in order from a-side to b-side
        outer_nodes = [a] + [onid for (t, onid, inid) in seams] + [b]
        inner_nodes = [inner_left] + [inid for (t, onid, inid) in seams] + [inner_right]
        for i in range(len(outer_nodes) - 1):
            final_elements.append([outer_nodes[i], outer_nodes[i + 1],
                                    inner_nodes[i + 1], inner_nodes[i]])

    # defaultdict(set) maps each node to the set of quad indices it belongs to
    # node_to_quads_v3: node -> set of quad indices (includes split-point nodes)
    node_to_quads_v3 = defaultdict(set)
    for qi, quad in enumerate(quad_quads):
        for n in quad:
            node_to_quads_v3[n].add(qi)
    for key, splits in edge_split.items():
        for _, nid in splits:
            for qi, _ in edge_to_quads.get(key, []):
                node_to_quads_v3[nid].add(qi)

    # dict.fromkeys() deduplicates interface nodes while preserving traversal order
    # interface_node_list_v3: ordered unique list of all nodes on the BL/far-field interface
    interface_node_list_v3 = list(dict.fromkeys(n for e in interface_edges for n in e))

    # interface_adj: adjacency map of the interface graph for BFS path-finding
    interface_adj = {}
    for e in interface_edges:
        a, b = e[0], e[1]
        interface_adj.setdefault(a, set()).add(b)
        interface_adj.setdefault(b, set()).add(a)

    def iface_path(start, end):
        """BFS shortest path through interface nodes from start to end."""
        if start == end:
            return [start]
        queue = deque([[start]])
        visited = {start}
        while queue:
            path = queue.popleft()
            for nb in interface_adj.get(path[-1], ()):
                if nb == end:
                    return path + [nb]
                if nb not in visited:
                    visited.add(nb)
                    queue.append(path + [nb])
        return None

    # tri_final: growing list of remapped/fan-split tri elements
    tri_final = []
    for tri in tri_elements:
        new_tri = [tri_node_mapping.get(nn, nn) for nn in tri]
        if len(set(new_tri)) < 3:
            continue
        snapped = [(i, new_tri[i]) for i in range(3) if tri[i] in tri_node_mapping]
        # fan_tris: replacement tris when the snapped edge spans multiple quad edges
        fan_tris = None
        for s in range(len(snapped)):
            for t in range(s + 1, len(snapped)):
                ei, ni = snapped[s]
                ej, nj = snapped[t]
                if node_to_quads_v3[ni].isdisjoint(node_to_quads_v3[nj]):
                    ek = 3 - ei - ej
                    nc = new_tri[ek]
                    path = iface_path(ni, nj)
                    if path and len(path) >= 2:
                        # fan-triangulate from nc over every edge in the interface path;
                        # handles 1 intermediate node (bisect) and N>1 (multi-cut) uniformly
                        fan_tris = [[nc, path[k], path[k + 1]] for k in range(len(path) - 1)]
                    else:
                        # fallback: closest single intermediate node when interface is disconnected
                        candidates = [n for n in interface_node_list_v3 if n != ni and n != nj]
                        if candidates:
                            p_nc = np.array(vertex_coords[nc])
                            cand_pts = np.array([vertex_coords[n] for n in candidates])
                            mid = candidates[int(np.argmin(np.linalg.norm(cand_pts - p_nc, axis=1)))]
                            fan_tris = [[nc, ni, mid], [nc, mid, nj]]
                    break
            if fan_tris is not None:
                break
        if fan_tris is not None:
            tri_final.extend(fan_tris)
        else:
            tri_final.append(new_tri)

    # tri_area() filters out degenerate (zero-area) triangles before adding to output
    tri_final = [
        t for t in tri_final
        if tri_area(vertex_coords[t[0]], vertex_coords[t[1]], vertex_coords[t[2]]) > 1e-14
    ]

    def signed_area(t):
        ax, ay = vertex_coords[t[0]]
        bx, by = vertex_coords[t[1]]
        cx, cy = vertex_coords[t[2]]
        return (bx - ax) * (cy - ay) - (cx - ax) * (by - ay)

    # signed_area() ensures all tris have consistent counter-clockwise orientation
    tri_final = [
        t if signed_area(t) > 0 else [t[0], t[2], t[1]]
        for t in tri_final
    ]
    final_elements.extend(tri_final)
    return final_elements


def merge_renumber_export(
    final_elements, quad_boundary, tri_boundary, vertex_coords,
    edge_split, quad_split, farfield_bc, tri_interface_bc, output_file,
):
    """Merge boundary records, renumber nodes (tris first), export to .vol, and print summary."""
    # merged_boundary: combined boundary dict for the hybrid mesh (quad wall + tri far-field)
    merged_boundary = {}
    # idx: running key counter for merged_boundary entries
    idx = 0

    for val in quad_boundary.values():
        if val[0] == farfield_bc:
            continue
        # n2, n3: the two node IDs of this boundary edge record
        n2, n3   = val[2], val[3]
        # wall_key: canonical (min, max) edge key used to look up any split on this wall edge
        wall_key = (min(n2, n3), max(n2, n3))
        if wall_key in edge_split:
            # d_start/d_end: NGSolve arc-length dist values at the two endpoints
            d_start = val[9]
            d_end   = val[11]
            # Build ordered waypoints from n2 to n3, interpolating dist at each split
            splits = edge_split[wall_key]  # sorted by canonical t
            if n2 == wall_key[0]:
                # n2 at canonical t=0: splits are in n2->n3 order
                mid_waypoints = [(nid, d_start + t * (d_end - d_start))
                                 for t, nid in splits]
            else:
                # n2 at canonical t=1: splits come in reverse order
                mid_waypoints = [(nid, d_start + (1.0 - t) * (d_end - d_start))
                                 for t, nid in reversed(splits)]
            waypoints = [(n2, d_start)] + mid_waypoints + [(n3, d_end)]
            for i in range(len(waypoints) - 1):
                sub = val[:]
                sub[2], sub[3]  = waypoints[i][0],  waypoints[i + 1][0]
                sub[9], sub[11] = waypoints[i][1],   waypoints[i + 1][1]
                merged_boundary[idx] = sub; idx += 1
        else:
            merged_boundary[idx] = val; idx += 1

    for val in tri_boundary.values():
        if val[0] == tri_interface_bc:
            continue
        merged_boundary[idx] = val; idx += 1

    # seen_tri: maps each tri-element node ID to its new compact index (tris numbered first)
    seen_tri  = {}
    # seen_quad: maps each quad-only node ID to its new compact index (after all tri nodes)
    seen_quad = {}

    for elm in final_elements:
        if len(elm) == 3:
            for nid in elm:
                if nid not in seen_tri:
                    seen_tri[nid] = len(seen_tri)

    for elm in final_elements:
        if len(elm) == 4:
            for nid in elm:
                if nid not in seen_tri and nid not in seen_quad:
                    seen_quad[nid] = len(seen_tri) + len(seen_quad)

    # old_to_new: merged mapping from old node IDs to new compact IDs (tri nodes first)
    old_to_new = {**seen_tri, **seen_quad}

    for val in merged_boundary.values():
        for nid in (val[2], val[3]):
            if nid not in old_to_new:
                old_to_new[nid] = len(old_to_new)

    # new_vertex_coords: compacted vertex coordinate list indexed by new compact node IDs
    new_vertex_coords = [None] * len(old_to_new)
    for old_id, new_id in old_to_new.items():
        new_vertex_coords[new_id] = vertex_coords[old_id]

    # remap all element node IDs to compact new IDs
    final_elements = [[old_to_new[n] for n in elm] for elm in final_elements]

    for val in merged_boundary.values():
        val[2] = old_to_new[val[2]]
        val[3] = old_to_new[val[3]]

    # exportMesh1() writes the hybrid mesh to a .vol file in NGSolve format
    exportMesh1(output_file, final_elements, merged_boundary, new_vertex_coords)

    # n_quads: count of quadrilateral elements in the final mesh
    n_quads = sum(1 for e in final_elements if len(e) == 4)
    # n_tris: count of triangular elements in the final mesh
    n_tris  = sum(1 for e in final_elements if len(e) == 3)
    print("------------------------------------------------")
    print("Hybrid mesh stitching v3 complete")
    print(f"  Quad elements  : {n_quads}")
    print(f"  Tri elements   : {n_tris}")
    print(f"  Total elements : {len(final_elements)}")
    print(f"  BL seams added : {len(quad_split)}")
    print(f"  Tri nodes      : {len(seen_tri)}")
    print(f"  Quad-only nodes: {len(seen_quad)}")
    print(f"Total nodes: {len(new_vertex_coords)}")
    print("------------------------------------------------")

    return final_elements, new_vertex_coords


def stitch_hybrid_mesh_v3(
    quad_mesh_file,
    tri_mesh_file,
    tri_interface_bc,
    farfield_bc=1,
    output_file="hybrid.vol",
    corner_tol=1e-1,
    in2d_file=None,
    wall_bc=2,
):
    """
    Stitch a quad BL mesh with a tri far-field mesh.

    For each tri interface node whose projection falls in the interior of a
    quad interface edge, a split seam is extended all the way through the
    boundary-layer column to the wall.  Every quad the seam passes through is
    split into two quads (no triangles are created in the BL region).

    Corner-snapped nodes (projection at t~0 or t~1) are handled the same as
    in v2 (no column split needed).

    If in2d_file is given, wall split nodes are placed on the exact rational
    quadratic Bezier geometry (formula A.3) using theta = the seam fraction
    extended straight from the BL interface.  Split wall boundary edges are
    recorded with theta-corrected dist values so NGSolve maps them to the
    correct position on the geometry segment.

    Output nodes are numbered: tri-element nodes first, then quad-only nodes.
    """
    # read_mesh() loads all elements, boundary data, and vertex coords from the quad BL mesh file
    # quad_elements: all elements from the quad BL mesh (may include tris and quads)
    # quad_boundary: boundary condition dict for the quad mesh
    # quad_vertices: list of [x, y] vertex coordinates for the quad mesh
    quad_elements, quad_boundary, quad_vertices = read_mesh(quad_mesh_file)
    # read_mesh() loads all elements, boundary data, and vertex coords from the tri far-field mesh file
    # tri_elements: all elements from the tri far-field mesh
    # tri_boundary: boundary condition dict for the tri mesh
    # tri_vertices: list of [x, y] vertex coordinates for the tri mesh
    tri_elements,  tri_boundary,  tri_vertices  = read_mesh(tri_mesh_file)

    # quad_tris: triangular elements embedded in the quad mesh (len == 3)
    quad_tris  = [e for e in quad_elements if len(e) == 3]
    # quad_quads: pure quadrilateral elements from the quad mesh (len == 4)
    quad_quads = [e for e in quad_elements if len(e) == 4]

    # extract_bl_interface_nodes() finds all free edges shared between the tri and quad regions
    # interface_edges: list of (n1, n2) edge pairs along the BL/far-field interface
    interface_edges = extract_bl_interface_nodes(quad_tris, quad_quads, quad_vertices)
    if not interface_edges:
        raise ValueError(
            "No interface edges found on the quad mesh. "
            "Check that the quad file has free edges not covered by boundary BCs."
        )

    wall_spline_map = None
    if in2d_file is not None:
        # read_in2d_wall_geometry() returns ordered [x, y] wall geometry points for arc-length projection
        # wall_edge_keys: canonical edge keys for all quad boundary edges tagged with wall_bc
        wall_edge_keys = {
            (min(val[2], val[3]), max(val[2], val[3]))
            for val in quad_boundary.values()
            if val[0] == wall_bc
        }
        # parse_in2d_spline_segments() extracts (p0, p_ctrl, p2) Bezier triples for each wall segment
        # spline_segs: list of (p0, p_ctrl, p2) numpy-array triples for each rational quadratic wall segment
        spline_segs     = parse_in2d_spline_segments(in2d_file, bc_flag=wall_bc)
        # build_wall_spline_map() matches each Bezier segment to its mesh wall edge by coordinate proximity
        wall_spline_map = build_wall_spline_map(spline_segs, quad_boundary, wall_bc)

    # vertex_coords: combined coordinate list -- quad vertices first, tri vertices appended after
    vertex_coords = [v[:] for v in quad_vertices]
    # tri_offset: index shift added to all tri node IDs so they do not collide with quad node IDs
    tri_offset    = len(quad_vertices)
    for v in tri_vertices:
        vertex_coords.append(v[:])

    # shift all tri element node indices up by tri_offset to make them global IDs
    tri_elements = [[n + tri_offset for n in e] for e in tri_elements]
    for k in tri_boundary:
        tri_boundary[k][2] += tri_offset
        tri_boundary[k][3] += tri_offset

    # get_boundary_edges() returns all boundary edges tagged with tri_interface_bc
    # tri_iface_edges: list of (n1, n2) edge pairs on the tri mesh side of the interface
    tri_iface_edges = get_boundary_edges(tri_boundary, tri_interface_bc)
    # edges_to_nodes() flattens edge pairs to a node list; dict.fromkeys() deduplicates while preserving order
    # tri_nodes: ordered unique list of tri interface node IDs
    tri_nodes       = list(dict.fromkeys(edges_to_nodes(tri_iface_edges)))

    # iface_edges_data: each interface edge enriched with numpy arrays for projection arithmetic
    iface_edges_data = [
        (a, b, np.array(vertex_coords[a]), np.array(vertex_coords[b]))
        for (a, b) in interface_edges
    ]

    # defaultdict(list) maps each canonical edge key to the list of (quad_idx, local_edge_idx) owners
    # edge_to_quads: adjacency dict from edge key to quads that contain that edge
    edge_to_quads = defaultdict(list)
    for qi, quad in enumerate(quad_quads):
        for li in range(4):
            a, b = quad[li], quad[(li + 1) % 4]
            # key: canonical (min, max) edge identifier shared by any quad owning this edge
            key  = (min(a, b), max(a, b))
            edge_to_quads[key].append((qi, li))
            #e.g.:edge_to_quads[8,9]={(0,1),(2,3)}
            #0,2->quad indices(qi);1,3->local edge index of the quads with edge[8,9]

    # project_and_trace_seams() projects tri interface nodes onto BL edges, then traces split seams inward
    # edge_split: maps each split edge key to (canonical_t, new_node_id)
    # tri_node_mapping: maps each tri interface node to its snapped/projected quad node
    # quad_split: maps each split quad index to (local_edge_idx, outer_nid, inner_nid)
    edge_split, tri_node_mapping, quad_split = project_and_trace_seams(
        tri_nodes, iface_edges_data, vertex_coords, edge_to_quads, quad_quads,
        corner_tol, wall_spline_map=wall_spline_map,
    )

    # build_elements() splits quads along seams, remaps/bisects tris, and assembles all output elements
    # final_elements: complete list of elements (quads + tris) for the stitched hybrid mesh
    final_elements = build_elements(
        quad_quads, tri_elements, tri_node_mapping, quad_split,
        edge_split, edge_to_quads, interface_edges, vertex_coords,
    )

    # merge_renumber_export() compacts node IDs, merges boundary dicts, and writes the .vol output file
    # final_elements: re-indexed element list after node compaction
    # new_vertex_coords: compacted vertex coordinate list (only nodes actually used)
    final_elements, new_vertex_coords = merge_renumber_export(
        final_elements, quad_boundary, tri_boundary, vertex_coords,
        edge_split, quad_split, farfield_bc, tri_interface_bc, output_file,
    )

    return final_elements, new_vertex_coords, tri_node_mapping


In [4]:
elem, bdry_data, vert = read_mesh("temp_bdry_hybrid (1).vol")
quad = [ele for ele in elem if len(ele) == 4]
tri = [ele for ele in elem if len(ele) == 3]
interface_edges = extract_bl_interface_nodes(tri, quad, vert)
print(interface_edges)
print(len(interface_edges))
print(interface_edges)
print(len(interface_edges))

[[0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9], [9, 10], [10, 11], [11, 12], [12, 13], [13, 14], [14, 15], [15, 160], [160, 16], [16, 161], [161, 17], [17, 162], [162, 18], [18, 163], [163, 19], [19, 20], [20, 21], [21, 22], [22, 164], [164, 23], [23, 165], [165, 24], [24, 25], [25, 26], [26, 27], [27, 28], [28, 29], [29, 30], [30, 31], [31, 32], [32, 33], [33, 34], [34, 35], [35, 36], [36, 166], [166, 37], [37, 167], [167, 38], [38, 168], [168, 39], [39, 169], [169, 40], [40, 170], [170, 41], [41, 171], [171, 42], [42, 172], [172, 43], [43, 173], [173, 44], [44, 45], [45, 46], [46, 47], [47, 48], [48, 49], [49, 50], [50, 51], [51, 52], [52, 53], [53, 54], [54, 55], [55, 174], [174, 56], [56, 57], [57, 58], [58, 59], [59, 60], [60, 61], [61, 175], [175, 62], [62, 176], [176, 63], [63, 177], [177, 64], [64, 178], [178, 65], [65, 66], [66, 179], [179, 67], [67, 68], [68, 69], [69, 70], [70, 71], [71, 72], [72, 73], [73, 74], [74, 75], [75, 76], [76, 77], [77, 78]

In [6]:
stitch_hybrid_mesh_v3(
    quad_mesh_file  = "temp_bl (1).vol",
    tri_mesh_file   = "temp_adap (1).vol",
    tri_interface_bc = 2,
    farfield_bc=1,
    output_file     = "stitch_new.vol",
    corner_tol=1e-1,
    in2d_file="netgen.in2d",
    wall_bc=2,
)



------------------------------------------------
Hybrid mesh stitching v3 complete
  Quad elements  : 637
  Tri elements   : 5207
  Total elements : 5844
  BL seams added : 35
  Tri nodes      : 2698
  Quad-only nodes: 638
Total nodes: 3336
------------------------------------------------


([[2698, 2699, 2700, 2701],
  [2699, 2702, 2703, 2700],
  [2702, 2704, 2705, 2703],
  [2704, 2706, 2707, 2705],
  [2701, 2708, 2709, 2698],
  [2710, 2711, 2712, 2713],
  [2711, 2709, 2708, 2712],
  [2713, 2714, 2715, 2710],
  [2714, 2716, 2717, 2715],
  [2716, 2718, 2719, 2717],
  [2718, 2720, 2721, 2719],
  [2720, 2722, 2723, 2721],
  [2722, 2724, 2725, 2723],
  [2724, 2726, 2727, 2725],
  [2726, 2728, 2729, 2727],
  [2728, 2730, 2731, 2729],
  [2730, 2732, 2733, 2731],
  [2732, 2734, 2735, 2733],
  [2734, 2736, 2737, 2735],
  [2736, 2738, 2739, 2737],
  [2738, 2740, 2741, 2739],
  [2740, 2742, 2743, 2741],
  [2742, 2744, 2745, 2743],
  [2744, 2746, 2747, 2745],
  [2746, 2748, 2749, 2747],
  [2748, 2750, 2751, 2749],
  [2752, 2753, 2754, 2755],
  [2753, 2751, 2750, 2754],
  [2755, 2756, 2757, 2752],
  [2756, 2758, 2759, 2757],
  [2758, 2760, 2761, 2759],
  [2760, 2762, 2763, 2761],
  [2762, 2764, 2765, 2763],
  [2764, 2766, 2767, 2765],
  [2766, 2768, 2769, 2767],
  [2768, 2770, 2771,

In [16]:
meshfile = "netgen_hybrid.vol"
metricfile = "adj_metric_graded.mtr"

element_vertex_id, boundary_data, vertex_coords = read_mesh(meshfile)
tri = [elm for elm in element_vertex_id if len(elm) == 3]
quad = [elm for elm in element_vertex_id if len(elm) == 4]
print(len(tri))
print(len(quad))
tri2 = quads_to_tris(quad)
tri.extend(tri2)
print(len(tri))
exportMesh1("netgen_tri.vol", tri, boundary_data, vertex_coords)


1582
574
2730


In [17]:
bcFlag = 2
element_node_id, boundary_data, vertex_coords, nodal_metric = read_mesh(meshfile, metricfile)
tri_id = [elm for elm in element_node_id if len(elm) == 3]
quad_id = [elm for elm in element_node_id if len(elm) == 4]
interface_edges = extract_bl_interface_nodes(tri_id, quad_id, vertex_coords)
tri_nodes_set = set(node for elm in tri_id for node in elm)
vertex_coords = vertex_coords[:len(tri_nodes_set)]
nodal_metric = nodal_metric[:len(tri_nodes_set)]

with open("temp.mtr", "w") as f:
    f.write(f"{len(nodal_metric)} {3}\n")
    # metric entries
    for metric in nodal_metric:
        f.write(" ".join(f"{val: .16e}" for val in metric) + "\n")

ordered_boundary_data = order_boundary_data(boundary_data, bcFlag)
for i in range(len(interface_edges)):
    ordered_boundary_data[i][2] = interface_edges[i][0]
    ordered_boundary_data[i][3] = interface_edges[i][1]
    ordered_boundary_data[i][9] = 0
    ordered_boundary_data[i][11] = 1

# Read farfield point coordinates from netgen.in2d (bc=1), then match to vertex_coords node IDs
farfield_pts = read_in2d_wall_geometry("netgen.in2d", bc_flag=1)
vc_arr = np.array(vertex_coords)
farfield_node_ids = [int(np.argmin(np.linalg.norm(vc_arr - np.array(pt), axis=1))) for pt in farfield_pts]
farfield_edges = [[farfield_node_ids[i], farfield_node_ids[i + 1]] for i in range(len(farfield_node_ids) - 1)]
farfield_edges.append([farfield_node_ids[-1], farfield_node_ids[0]])  # close the loop

exportMesh1("temp.vol", tri_id, ordered_boundary_data, vertex_coords)
write_in2d("temp1.in2d", interface_edges, farfield_edges, bcFlag, 1, vertex_coords)

# write boundary_data values and ordered_boundary_data values to temporary file
# with open("tmp_boundary_check.dat", "w") as f:
#     for key, val in boundary_data.items():
#         f.write(f"{val}\n")

.in2d file written to: temp1.in2d


In [18]:
bcFlag = 2
delta0 = 1e-3
growthFactor = 1.2
blLayers = 30
blHeight = 0.0129159
#meshFile = "temp_bdry.vol"

geometryFile = r"E:\hybrid_adaptation\copy\aflr_abhigyan_copy\src\meshAdapt\netgen.in2d"
bgmeshFile = r"E:\hybrid_adaptation\copy\aflr_abhigyan_copy\src\meshAdapt\netgen_tri_prev.vol"
metricFile = r"E:\hybrid_adaptation\copy\aflr_abhigyan_copy\src\meshAdapt\adj_metric_graded.mtr"
bgMesh = Triangulation.fromMesh(geometryFile, bgmeshFile, metricFile)
"""
element_vertex_id, boundary_data, vertex_coords = read_mesh(meshFile)
ordered_boundary_data = order_boundary_data(boundary_data, bcFlag)

get viscous edges from boundary_data whose first entry is bcFlag
viscous_edges_id = [[edge[2], edge[3]] for edge in ordered_boundary_data.values() if edge[0] == bcFlag]
farfield_edges_id = [[edge[2], edge[3]] for edge in ordered_boundary_data.values() if edge[0] == 1]
print(f"Viscous edges: {viscous_edges_id}")
print(f"Number of viscous edges: {len(viscous_edges_id)}")
print(f"Farfield edges: {farfield_edges_id}")
print(f"Number of Farfield edges: {len(farfield_edges_id)}")

print(f"Length of vertex_coords before deformation: {len(vertex_coords)}")
vertex_coords, new_quads, updated_viscous_edges, blLayerHeights = quad_layer(bgMesh, viscous_edges_id, farfield_edges_id, vertex_coords, blLayers, blHeight, delta0 = None, growthFactor = None)
print(f"Length of vertex_coords after deformation: {len(vertex_coords)}")
print(f"Length of new_quads: {len(new_quads)}")
print(f"New quads: {new_quads}")
print(f"Length of updated_viscous_edges: {len(updated_viscous_edges)}")
print(f"Updated viscous edges: {updated_viscous_edges}")
print(f"Boundary layer heights: {blLayerHeights}")
print(f"Total height from blLayerHeights: {sum(blLayerHeights)}")
"""

'\nelement_vertex_id, boundary_data, vertex_coords = read_mesh(meshFile)\nordered_boundary_data = order_boundary_data(boundary_data, bcFlag)\n\nget viscous edges from boundary_data whose first entry is bcFlag\nviscous_edges_id = [[edge[2], edge[3]] for edge in ordered_boundary_data.values() if edge[0] == bcFlag]\nfarfield_edges_id = [[edge[2], edge[3]] for edge in ordered_boundary_data.values() if edge[0] == 1]\nprint(f"Viscous edges: {viscous_edges_id}")\nprint(f"Number of viscous edges: {len(viscous_edges_id)}")\nprint(f"Farfield edges: {farfield_edges_id}")\nprint(f"Number of Farfield edges: {len(farfield_edges_id)}")\n\nprint(f"Length of vertex_coords before deformation: {len(vertex_coords)}")\nvertex_coords, new_quads, updated_viscous_edges, blLayerHeights = quad_layer(bgMesh, viscous_edges_id, farfield_edges_id, vertex_coords, blLayers, blHeight, delta0 = None, growthFactor = None)\nprint(f"Length of vertex_coords after deformation: {len(vertex_coords)}")\nprint(f"Length of new_q

In [ ]:
'''for i in range(len(updated_viscous_edges)):
    ordered_boundary_data[i][2] = updated_viscous_edges[i][0]
    ordered_boundary_data[i][3] = updated_viscous_edges[i][1]

# append new_quads to element_vertex_id
element_vertex_id.extend(new_quads)
exportMesh1("temp_bdry_hybrid.vol", element_vertex_id, ordered_boundary_data, vertex_coords)
'''

In [6]:
def join_meshes(tri_mesh_file, quad_mesh_file, output_file="joined.vol"):
    """
    Concatenate a triangular mesh file and a quad mesh file into one .vol file.
    Triangles are stripped from the quad mesh before joining.
    Node lists are merged by appending quad mesh nodes after tri mesh nodes.
    No interface matching or deduplication is performed.
    """
    tri_elements, tri_boundary, tri_vertices = read_mesh(tri_mesh_file)
    quad_elements, quad_boundary, quad_vertices = read_mesh(quad_mesh_file)

    quad_only = [elm for elm in quad_elements if len(elm) == 4]

    offset = len(tri_vertices)
    combined_vertices = tri_vertices + quad_vertices

    offset_quad_elements = [[n + offset for n in elm] for elm in quad_only]
    combined_elements = tri_elements + offset_quad_elements

    combined_boundary = {}
    idx = 0
    for val in tri_boundary.values():
        combined_boundary[idx] = val[:]
        idx += 1
    for val in quad_boundary.values():
        entry = val[:]
        entry[2] += offset
        entry[3] += offset
        combined_boundary[idx] = entry
        idx += 1

    exportMesh1(output_file, combined_elements, combined_boundary, combined_vertices)

    n_tris  = sum(1 for e in combined_elements if len(e) == 3)
    n_quads = sum(1 for e in combined_elements if len(e) == 4)
    print("------------------------------------------------")
    print("Mesh join complete")
    print(f"  Tri elements   : {n_tris}")
    print(f"  Quad elements  : {n_quads}")
    print(f"  Total elements : {len(combined_elements)}")
    print(f"  Total nodes    : {len(combined_vertices)}")
    print("------------------------------------------------")

    return combined_elements, combined_boundary, combined_vertices


# Example usage:
# join_meshes("tri_mesh.vol", "quad_mesh.vol", output_file="joined.vol")


In [4]:
# ============================================================
# Uniform triangle refinement with Bezier-aware boundary midpoints
# ============================================================
# Each triangle is split 1->4 by inserting edge midpoints.
# Edges that lie on the quadratic Bezier boundary layer use the
# rational quadratic parametric form (formula A.3) to place the
# midpoint ON the curve.  All other edges use the arithmetic midpoint.
#
# Rational quadratic Bezier (weight w derived from control points):
#   B(t) = [ (1-t)^2 P0 + w*t*(1-t) P_ctrl + t^2 P2 ]
#          / [ (1-t)^2 + w*t*(1-t)        + t^2      ]
#
# Weight:  w = ||P0 - P2|| / sqrt(0.5*(||P0-P_ctrl||^2 + ||P2-P_ctrl||^2))
# ============================================================

def raz_weight(P0, P_ctrl, P2):
    """Rational quadratic Bezier weight from control-point geometry."""
    chord = np.linalg.norm(P2 - P0)
    d     = np.sqrt(0.5 * (np.linalg.norm(P0 - P_ctrl)**2
                           + np.linalg.norm(P2 - P_ctrl)**2))
    return chord / d if d > 1e-14 else 1.0


def raz_eval(P0, P_ctrl, P2, t, w):
    """Evaluate rational quadratic Bezier at parameter t."""
    denom = (1 - t)**2 + w * t * (1 - t) + t**2
    return ((1 - t)**2 * P0 + w * t * (1 - t) * P_ctrl + t**2 * P2) / denom


def raz_parameter(point, P0, P_ctrl, P2, w, tol=1e-8):
    """
    Find t in [0, 1] such that B(t) = point on the rational quadratic Bezier.

    Substituting B(t) = point and cross-multiplying gives a quadratic per
    coordinate:

        (A + D - w*C)*t^2  +  (-2*A + w*C)*t  +  A  =  0

    where  A = P0[d] - point[d],  C = P_ctrl[d] - point[d],
           D = P2[d]  - point[d].

    Returns the parameter t if found, or None if the point is not on this
    segment within the given tolerance.
    """
    point = np.asarray(point, float)
    candidates = []
    for dim in range(2):
        A  = P0[dim]     - point[dim]
        C  = P_ctrl[dim] - point[dim]
        D  = P2[dim]     - point[dim]
        ca = A + D - w * C
        cb = -2.0 * A + w * C
        cc = A
        if abs(ca) < 1e-14:
            if abs(cb) > 1e-14:
                candidates.append(-cc / cb)
        else:
            disc = cb * cb - 4.0 * ca * cc
            if disc >= 0.0:
                sq = np.sqrt(max(disc, 0.0))
                candidates += [(-cb + sq) / (2.0 * ca),
                               (-cb - sq) / (2.0 * ca)]

    for t in candidates:
        if -tol <= t <= 1.0 + tol:
            t_c = np.clip(t, 0.0, 1.0)
            if np.linalg.norm(raz_eval(P0, P_ctrl, P2, t_c, w) - point) < tol:
                return t_c
    return None


def build_bl_edge_map(spline_segs, boundary_data, vertex_coords,
                       bl_bc, tol=1e-8):
    """
    Match boundary edges (bc == bl_bc) to rational quadratic Bezier segments.

    For each matching edge the map stores
      (min_n, max_n) -> (P0_can, P_ctrl, P2_can, w, t0, t1)
    where t=0 is at vertex key[0] and t=1 is at vertex key[1].

    Parameters
    ----------
    spline_segs   : list of (P0, P_ctrl, P2) from parse_in2d_spline_segments
    boundary_data : dict as returned by read_mesh
    vertex_coords : list of [x, y]
    bl_bc         : BC flag identifying the curved boundary
    tol           : geometric tolerance for on-curve detection

    Returns
    -------
    dict  (min_n, max_n) -> (P0_can, P_ctrl, P2_can, w, t0, t1)
    """
    bl_edge_keys = {
        (min(val[2], val[3]), max(val[2], val[3])): (val[2], val[3])
        for val in boundary_data.values()
        if val[0] == bl_bc
    }

    bezier_edge_map = {}
    for (P0, P_ctrl, P2) in spline_segs:
        P0     = np.asarray(P0,     float)
        P_ctrl = np.asarray(P_ctrl, float)
        P2     = np.asarray(P2,     float)
        w = raz_weight(P0, P_ctrl, P2)

        for key in bl_edge_keys:
            if key in bezier_edge_map:
                continue
            v0 = np.array(vertex_coords[key[0]], float)
            v1 = np.array(vertex_coords[key[1]], float)
            t0 = raz_parameter(v0, P0, P_ctrl, P2, w, tol)
            t1 = raz_parameter(v1, P0, P_ctrl, P2, w, tol)
            if t0 is not None and t1 is not None:
                bezier_edge_map[key] = (P0, P_ctrl, P2, w, t0, t1)

    return bezier_edge_map


def refine_triangles(vertex_coords, triangles, boundary_data,
                     bezier_edge_map=None):
    """
    Uniform 1->4 triangle refinement with Bezier-aware edge midpoints.

    Each triangle is split into 4 children by inserting midpoints on its
    three edges.  For edges recorded in bezier_edge_map (edges lying on
    the quadratic Bezier boundary layer) the midpoint is placed on the
    curve using the rational parametric form.  All other edges use the
    arithmetic midpoint.

    Child layout (i0, i1, i2 = original corners; m01, m12, m02 = midpoints):

        [i0,  m01, m02]   corner at i0
        [m01, i1,  m12]   corner at i1
        [m02, m12, i2 ]   corner at i2
        [m01, m12, m02]   centre

    Boundary edges are split at their midpoints.  The dist values in the
    boundary records are linearly interpolated (theta = 0.5 at the
    midpoint) so NGSolve maps each sub-edge to the correct position on
    the geometry segment.

    Parameters
    ----------
    vertex_coords   : list of [x, y]
    triangles       : list of [n0, n1, n2] (0-based)
    boundary_data   : dict as returned by read_mesh (not mutated in-place)
    bezier_edge_map : dict from build_bl_edge_map, or None

    Returns
    -------
    new_vertex_coords : list of [x, y]
    new_triangles     : list of [n0, n1, n2]
    new_boundary_data : dict with split boundary edge records
    """
    nodes           = np.array(vertex_coords, dtype=float)
    bezier_edge_map = bezier_edge_map or {}
    new_nodes       = [v[:] for v in vertex_coords]
    edge_to_mid     = {}   # canonical key -> new node index

    def get_mid(i, j):
        key = (min(i, j), max(i, j))
        if key in edge_to_mid:
            return edge_to_mid[key]

        if key in bezier_edge_map:
            P0, P_ctrl, P2, w, t0, t1 = bezier_edge_map[key]
            t_mid = (t0 + t1) * 0.5
            mid   = raz_eval(P0, P_ctrl, P2, t_mid, w).tolist()
        else:
            mid = ((nodes[i] + nodes[j]) * 0.5).tolist()

        mid_idx = len(new_nodes)
        new_nodes.append(mid)
        edge_to_mid[key] = mid_idx
        return mid_idx

    new_tris = []
    for i0, i1, i2 in triangles:
        m01 = get_mid(i0, i1)
        m12 = get_mid(i1, i2)
        m02 = get_mid(i0, i2)
        new_tris.extend([
            [i0,  m01, m02],
            [m01, i1,  m12],
            [m02, m12, i2 ],
            [m01, m12, m02],
        ])

    # ----------------------------------------------------------------
    # Update boundary data: split every boundary edge that was refined.
    # dist is linearly interpolated; theta = 0.5 always at the midpoint.
    # ----------------------------------------------------------------
    new_boundary = {}
    bdry_idx = 0
    for val in boundary_data.values():
        n2, n3 = val[2], val[3]
        key    = (min(n2, n3), max(n2, n3))

        if key not in edge_to_mid:
            new_boundary[bdry_idx] = val[:]
            bdry_idx += 1
            continue

        mid_idx = edge_to_mid[key]
        d_start = val[9]
        d_end   = val[11]
        d_mid   = (d_start + d_end) * 0.5

        sub_a       = val[:]
        sub_b       = val[:]
        sub_a[2],  sub_a[3]  = n2,      mid_idx
        sub_a[9],  sub_a[11] = d_start, d_mid
        sub_b[2],  sub_b[3]  = mid_idx, n3
        sub_b[9],  sub_b[11] = d_mid,   d_end

        new_boundary[bdry_idx] = sub_a; bdry_idx += 1
        new_boundary[bdry_idx] = sub_b; bdry_idx += 1

    return new_nodes, new_tris, new_boundary


def uniform_refine_mesh(mesh_file, output_file,
                        in2d_file=None, bl_bc=2,
                        n_levels=1, tol=1e-8):
    """
    Uniformly refine the triangles of a mesh and write the result.

    Each refinement level splits every triangle into 4 children (1->4).
    For triangles that have an edge on the Bezier boundary layer (bc==bl_bc,
    described by 3-point segments in in2d_file), the midpoint of that edge
    is placed on the rational quadratic Bezier curve rather than on the
    chord, keeping the refined mesh on the true geometry.

    Quad elements are passed through unchanged; only triangles are refined.
    Boundary edge records are split and dist values are linearly interpolated.

    Parameters
    ----------
    mesh_file   : str   input .vol file
    output_file : str   output .vol file
    in2d_file   : str or None
                  Netgen .in2d file with 3-point Bezier wall segments.
                  Pass None to use arithmetic midpoints for all edges.
    bl_bc       : int   BC flag of the curved boundary (default 2)
    n_levels    : int   number of refinement levels (default 1)
    tol         : float geometric tolerance for on-curve detection

    Returns
    -------
    elements      : list of element connectivity (tris then quads)
    boundary_data : updated boundary dict
    vertex_coords : updated coordinate list

    Example
    -------
    uniform_refine_mesh("temp.vol", "temp_refined.vol",
                        in2d_file="netgen.in2d", bl_bc=2, n_levels=2)
    """
    elements, boundary_data, vertex_coords = read_mesh(mesh_file)

    triangles = [e for e in elements if len(e) == 3]
    quads     = [e for e in elements if len(e) == 4]

    # Build Bezier edge map from the in2d file for the curved boundary
    bezier_edge_map = {}
    spline_segs     = []
    if in2d_file is not None:
        spline_segs = parse_in2d_spline_segments(in2d_file, bc_flag=bl_bc)
        if spline_segs:
            bezier_edge_map = build_bl_edge_map(
                spline_segs, boundary_data, vertex_coords, bl_bc, tol
            )

    for _level in range(n_levels):
        vertex_coords, triangles, boundary_data = refine_triangles(
            vertex_coords, triangles, boundary_data, bezier_edge_map
        )
        # After each level the boundary edges have been split; rebuild the
        # map so that the new sub-edges on the Bezier are also curved.
        if spline_segs:
            bezier_edge_map = build_bl_edge_map(
                spline_segs, boundary_data, vertex_coords, bl_bc, tol
            )

    all_elements = triangles + quads
    exportMesh1(output_file, all_elements, boundary_data, vertex_coords)

    print("------------------------------------------------")
    print("Uniform triangle refinement complete")
    print(f"  Refinement levels : {n_levels}")
    print(f"  Triangles         : {len(triangles)}")
    print(f"  Quads (unchanged) : {len(quads)}")
    print(f"  Total elements    : {len(all_elements)}")
    print(f"  Total nodes       : {len(vertex_coords)}")
    if bezier_edge_map:
        print(f"  Bezier BL edges   : {len(bezier_edge_map)} (bc={bl_bc})")
    else:
        print("  No Bezier segments found; arithmetic midpoints used for all edges")
    print("------------------------------------------------")

    return all_elements, boundary_data, vertex_coords


In [8]:
# ============================================================
# Generate netgen_quadratic.in2d with rational quadratic Bezier airfoil segments
# ============================================================
# Reuses read_in2d_wall_geometry (cell 3) for farfield parsing.
# ============================================================

def compute_tangents(pts):
    """Unit tangent vectors via natural cubic spline (arc-length parameterised)."""
    pts = np.asarray(pts, dtype=float)
    ds = np.sqrt(np.sum(np.diff(pts, axis=0) ** 2, axis=1))
    s = np.concatenate([[0.0], np.cumsum(ds)])
    cs_x = CubicSpline(s, pts[:, 0], bc_type='natural')
    cs_y = CubicSpline(s, pts[:, 1], bc_type='natural')
    dx = cs_x(s, 1)
    dy = cs_y(s, 1)
    T = np.column_stack([dx, dy])
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms = np.where(norms < 1e-14, 1.0, norms)
    return T / norms


def quadratic_control_pts(pts):
    """
    For each segment (pts[i], pts[i+1]), find the quadratic Bezier control
    point as the intersection of the tangent lines at the two endpoints.

    Solves: lam*T0 + mu*T1 = P1 - P0  =>  C = P0 + lam*T0
    Falls back to chord midpoint when tangents are parallel or intersection
    lies behind an endpoint (lam<=0 or mu<=0).
    """
    pts = np.asarray(pts, dtype=float)
    T = compute_tangents(pts)
    ctrls = []
    for i in range(len(pts) - 1):
        P0, P1 = pts[i], pts[i + 1]
        t0, t1 = T[i], T[i + 1]
        A = np.column_stack([t0, t1])
        b = P1 - P0
        det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
        if abs(det) > 1e-12:
            lam, mu = np.linalg.solve(A, b)
            if lam > 0 and mu > 0:
                C = P0 + lam * t0
                seg = P1 - P0
                alpha = np.dot(C - P0, seg) / (np.dot(seg, seg) + 1e-30)
                if not (0.05 < alpha < 0.95):
                    C = (P0 + P1) / 2
            else:
                C = (P0 + P1) / 2
        else:
            C = (P0 + P1) / 2
        ctrls.append(C)
    return np.array(ctrls)


def gen_quadratic_bezier_in2d(airfoil_upper, airfoil_lower,
                               in2d_file, output_file):
    """
    Build a splinecurves2dv2 .in2d file with rational quadratic Bezier
    airfoil segments (bc=2) and straight farfield segments (bc=1).

    Parameters
    ----------
    airfoil_upper : array-like, shape (N, 2)
        Upper surface points ordered LE -> TE.
    airfoil_lower : array-like, shape (N, 2)
        Lower surface points ordered LE -> TE (index 0 = LE, -1 = TE).
    in2d_file : str
        Reference .in2d file whose bc=1 segments supply the farfield.
    output_file : str
        Path to write the new .in2d file.

    Notes
    -----
    Airfoil path: lower TE -> LE -> upper TE (lower surface reversed).
    A straight segment closes the blunt trailing edge (upper TE -> lower TE).
    Uses read_in2d_wall_geometry with bc_flag=1 to extract farfield points.
    """
    upper = np.asarray(airfoil_upper, dtype=float)
    lower = np.asarray(airfoil_lower, dtype=float)

    # Build airfoil path: lower TE -> LE -> upper TE
    lower_rev   = lower[::-1]
    airfoil_pts = np.vstack([lower_rev, upper[1:]])
    ctrls       = quadratic_control_pts(airfoil_pts)
    n_seg       = len(airfoil_pts) - 1

    # Interleave: P0 C0 P1 C1 P2 ... Pn
    point_list = []
    for i in range(n_seg):
        point_list.append(airfoil_pts[i].tolist())
        point_list.append(ctrls[i].tolist())


    # Farfield via read_in2d_wall_geometry with bc_flag=1 (straight segments)
    ff_pts        = read_in2d_wall_geometry(in2d_file, bc_flag=1)
    n_ff          = len(ff_pts)          # closed polygon: #segs == #pts
    outer_start_0 = len(point_list)      # 0-based offset for farfield pts
    for xy in ff_pts:
        point_list.append(list(xy))

    # Build in2d lines
    out_lines = ["splinecurves2dv2", "1", "", "points"]
    for idx, (x, y) in enumerate(point_list):
        out_lines.append(f"{idx + 1}	{x:.15g}	{y:.15g}")

    out_lines += ["", "segments"]

    for i in range(n_seg):
        p1, p2 = 2 * i + 1, 2 * i + 2
        p3 = 2 * i + 3 if i < n_seg - 1 else 1  # sharp TE: close back to point 1
        out_lines.append(f"1	0	3	{p1}	{p2}	{p3}	-bc=2")


    for i in range(n_ff):
        p1 = outer_start_0 + i + 1
        p2 = outer_start_0 + (i + 1) % n_ff + 1
        out_lines.append(f"1	0	2	{p1}	{p2}	-bc=1")

    out_lines += ["", "materials", "1	domain1"]

    with open(output_file, "w") as fout:
        fout.write("\n".join(out_lines))


    print(f"Airfoil on-curve points : {len(airfoil_pts)}")
    print(f"Airfoil segments        : {n_seg} quadratic + 1 TE straight")
    print(f"Farfield points\segs    : {n_ff}")
    print(f"Total points in file    : {len(point_list)}")
    print(f"Written -> {output_file}")
    return point_list


<>:128: SyntaxWarning: invalid escape sequence '\s'
<>:128: SyntaxWarning: invalid escape sequence '\s'
C:\Users\abhi0\AppData\Local\Temp\ipykernel_14304\2170480855.py:128: SyntaxWarning: invalid escape sequence '\s'
  print(f"Farfield points\segs    : {n_ff}")


In [65]:
upper = np.array([
    [0.0000000000, 0.0000000000], [0.0005663304, 0.0041964396], [0.0022640387, 0.0083040018],
    [0.0050892791, 0.0123181710], [0.0090356514, 0.0162330903], [0.0140942158, 0.0200416597],
    [0.0202535132, 0.0237356696], [0.0274995906, 0.0273059662], [0.0358160335, 0.0307426424],
    [0.0451840023, 0.0340352519], [0.0555822757, 0.0371730371], [0.0669872981, 0.0401451679],
    [0.0793732336, 0.0429409830], [0.0927120240, 0.0455502280], [0.1069734526, 0.0479632838],
    [0.1221252128, 0.0501713806], [0.1381329809, 0.0521667906], [0.1549604943, 0.0539429965],
    [0.1725696330, 0.0554948310], [0.1909205069, 0.0568185853], [0.2099715452, 0.0579120842],
    [0.2296795913, 0.0587747280], [0.2500000000, 0.0594075000], [0.2708867391, 0.0598129427],
    [0.2922924935, 0.0599951028], [0.3141687722, 0.0599594501], [0.3364660183, 0.0597127721],
    [0.3591337216, 0.0592630494], [0.3821205322, 0.0586193171], [0.4053743778, 0.0577915153],
    [0.4288425809, 0.0567903353], [0.4524719783, 0.0556270653], [0.4762090421, 0.0543134407],
    [0.5000000000, 0.0528615020], [0.5237909579, 0.0512834656], [0.5475280217, 0.0495916088],
    [0.5711574191, 0.0477981716], [0.5946256222, 0.0459152775], [0.6178794678, 0.0439548727],
    [0.6408662784, 0.0419286847], [0.6635339817, 0.0398481978], [0.6858312278, 0.0377246460],
    [0.7077075065, 0.0355690188], [0.7291132609, 0.0333920795], [0.7500000000, 0.0312043904],
    [0.7703204087, 0.0290163444], [0.7900284548, 0.0268381970], [0.8090794931, 0.0246800970],
    [0.8274303670, 0.0225521128], [0.8450395057, 0.0204642502], [0.8618670191, 0.0184264607],
    [0.8778747872, 0.0164486377], [0.8930265474, 0.0145405997], [0.9072879760, 0.0127120589],
    [0.9206267664, 0.0109725773], [0.9330127019, 0.0093315085], [0.9444177243, 0.0077979285],
    [0.9548159977, 0.0063805560], [0.9641839665, 0.0050876652], [0.9725004094, 0.0039269939],
    [0.9797464868, 0.0029056497], [0.9859057842, 0.0020300172], [0.9909643486, 0.0013056699],
    [0.9949107209, 0.0007372892], [0.9977359613, 0.0003285947], [0.9994336696, 0.0000822858],
    [1.0000000000, 0.0000000000]
])  # 67 points

lower = np.array([
    [0.0000000000, 0.0000000000], [0.0005663304, -0.0041964396], [0.0022640387, -0.0083040018],
    [0.0050892791, -0.0123181710], [0.0090356514, -0.0162330903], [0.0140942158, -0.0200416597],
    [0.0202535132, -0.0237356696], [0.0274995906, -0.0273059662], [0.0358160335, -0.0307426424],
    [0.0451840023, -0.0340352519], [0.0555822757, -0.0371730371], [0.0669872981, -0.0401451679],
    [0.0793732336, -0.0429409830], [0.0927120240, -0.0455502280], [0.1069734526, -0.0479632838],
    [0.1221252128, -0.0501713806], [0.1381329809, -0.0521667906], [0.1549604943, -0.0539429965],
    [0.1725696330, -0.0554948310], [0.1909205069, -0.0568185853], [0.2099715452, -0.0579120842],
    [0.2296795913, -0.0587747280], [0.2500000000, -0.0594075000], [0.2708867391, -0.0598129427],
    [0.2922924935, -0.0599951028], [0.3141687722, -0.0599594501], [0.3364660183, -0.0597127721],
    [0.3591337216, -0.0592630494], [0.3821205322, -0.0586193171], [0.4053743778, -0.0577915153],
    [0.4288425809, -0.0567903353], [0.4524719783, -0.0556270653], [0.4762090421, -0.0543134407],
    [0.5000000000, -0.0528615020], [0.5237909579, -0.0512834656], [0.5475280217, -0.0495916088],
    [0.5711574191, -0.0477981716], [0.5946256222, -0.0459152775], [0.6178794678, -0.0439548727],
    [0.6408662784, -0.0419286847], [0.6635339817, -0.0398481978], [0.6858312278, -0.0377246460],
    [0.7077075065, -0.0355690188], [0.7291132609, -0.0333920795], [0.7500000000, -0.0312043904],
    [0.7703204087, -0.0290163444], [0.7900284548, -0.0268381970], [0.8090794931, -0.0246800970],
    [0.8274303670, -0.0225521128], [0.8450395057, -0.0204642502], [0.8618670191, -0.0184264607],
    [0.8778747872, -0.0164486377], [0.8930265474, -0.0145405997], [0.9072879760, -0.0127120589],
    [0.9206267664, -0.0109725773], [0.9330127019, -0.0093315085], [0.9444177243, -0.0077979285],
    [0.9548159977, -0.0063805560], [0.9641839665, -0.0050876652], [0.9725004094, -0.0039269939],
    [0.9797464868, -0.0029056497], [0.9859057842, -0.0020300172], [0.9909643486, -0.0013056699],
    [0.9949107209, -0.0007372892], [0.9977359613, -0.0003285947], [0.9994336696, -0.0000822858],
    [1.0000000000,  0.0000000000],
]) 




gen_quadratic_bezier_in2d(
    airfoil_upper = upper,
    airfoil_lower = lower,
    in2d_file     = "netgen.in2d",
    output_file   = "naca0.in2d",
)


Airfoil on-curve points : 133
Airfoil segments        : 132 quadratic + 1 TE straight
Farfield points\segs    : 80
Total points in file    : 344
Written -> naca0.in2d


[[1.0, 0.0],
 [0.999622447485202, -5.48646700535635e-05],
 [0.9994336696, -8.22858e-05],
 [0.9986069958298287, -0.00020236516655777818],
 [0.9977359613, -0.0003285947],
 [0.996316910821044, -0.0005342422073666513],
 [0.9949107209, -0.0007372892],
 [0.9929426580779822, -0.0010214679096777904],
 [0.9909643486, -0.0013056699],
 [0.9884387840081483, -0.0016684900207546553],
 [0.9859057842, -0.0020300172],
 [0.9828328167888595, -0.0024686122746970813],
 [0.9797464868, -0.0029056497],
 [0.9761321962674786, -0.0034174485790348357],
 [0.9725004094, -0.0039269939],
 [0.9683535679589254, -0.004508802175151086],
 [0.9641839665, -0.0050876652],
 [0.9595140227710591, -0.0057359904127184625],
 [0.9548159977, -0.006380556],
 [0.9496336165260426, -0.007091574888657314],
 [0.9444177243, -0.0077979285],
 [0.9387345618950187, -0.00856756140548113],
 [0.9330127019, -0.0093315085],
 [0.9268414531509979, -0.010155455180879595],
 [0.9206267664, -0.0109725773],
 [0.9139810777634804, -0.011846368594599907],
 [

In [34]:
# netgen.in2d has 3-point Bezier segments for bc=2
# midpoints of wall edges are placed ON the curve, not on the chord
uniform_refine_mesh(
    mesh_file   = "naca1.vol",
    output_file = "naca2.vol",
    in2d_file   = "naca.in2d",
    bl_bc       = 2,
    n_levels    = 1
)


------------------------------------------------
Uniform triangle refinement complete
  Refinement levels : 1
  Triangles         : 25312
  Quads (unchanged) : 0
  Total elements    : 25312
  Total nodes       : 12980
  Bezier BL edges   : 328 (bc=2)
------------------------------------------------


([[0, 3326, 3328],
  [3326, 872, 3327],
  [3328, 3327, 874],
  [3326, 3327, 3328],
  [872, 3329, 3331],
  [3329, 1, 3330],
  [3331, 3330, 873],
  [3329, 3330, 3331],
  [874, 3332, 3334],
  [3332, 873, 3333],
  [3334, 3333, 366],
  [3332, 3333, 3334],
  [872, 3331, 3327],
  [3331, 873, 3332],
  [3327, 3332, 874],
  [3331, 3332, 3327],
  [1, 3335, 3330],
  [3335, 875, 3336],
  [3330, 3336, 873],
  [3335, 3336, 3330],
  [875, 3337, 3339],
  [3337, 160, 3338],
  [3339, 3338, 876],
  [3337, 3338, 3339],
  [873, 3340, 3333],
  [3340, 876, 3341],
  [3333, 3341, 366],
  [3340, 3341, 3333],
  [875, 3339, 3336],
  [3339, 876, 3340],
  [3336, 3340, 873],
  [3339, 3340, 3336],
  [621, 3342, 3344],
  [3342, 877, 3343],
  [3344, 3343, 879],
  [3342, 3343, 3344],
  [877, 3345, 3347],
  [3345, 762, 3346],
  [3347, 3346, 878],
  [3345, 3346, 3347],
  [879, 3348, 3350],
  [3348, 878, 3349],
  [3350, 3349, 670],
  [3348, 3349, 3350],
  [877, 3347, 3343],
  [3347, 878, 3348],
  [3343, 3348, 879],
  [3347,

In [7]:
join_meshes("tempAdap.vol", "tempBL.vol", output_file="gap2.vol")


------------------------------------------------
Mesh join complete
  Tri elements   : 1410
  Quad elements  : 512
  Total elements : 1922
  Total nodes    : 1652
------------------------------------------------


([[562, 3, 583],
  [732, 4, 5],
  [13, 589, 14],
  [15, 589, 16],
  [17, 18, 15],
  [19, 18, 17],
  [20, 18, 21],
  [18, 19, 21],
  [26, 22, 27],
  [22, 28, 27],
  [557, 412, 22],
  [39, 767, 40],
  [44, 540, 43],
  [400, 319, 47],
  [603, 48, 49],
  [52, 603, 51],
  [54, 783, 55],
  [414, 64, 66],
  [243, 63, 414],
  [414, 63, 62],
  [65, 414, 66],
  [74, 364, 70],
  [71, 74, 69],
  [72, 74, 73],
  [75, 76, 77],
  [76, 377, 460],
  [81, 515, 82],
  [515, 83, 82],
  [83, 377, 78],
  [84, 85, 159],
  [84, 515, 86],
  [345, 360, 401],
  [429, 151, 96],
  [429, 95, 476],
  [97, 153, 476],
  [106, 107, 348],
  [107, 108, 348],
  [110, 111, 112],
  [113, 111, 114],
  [357, 283, 130],
  [132, 498, 129],
  [444, 508, 385],
  [405, 25, 22],
  [707, 31, 33],
  [411, 220, 459],
  [149, 396, 178],
  [476, 95, 98],
  [153, 152, 476],
  [154, 158, 103],
  [345, 513, 360],
  [113, 114, 160],
  [161, 112, 162],
  [164, 113, 160],
  [781, 468, 384],
  [706, 113, 167],
  [167, 113, 164],
  [586, 210, 1